### **CPU Quantization & Ablation Analysis**

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

from src.config import QUANTIZATION_DIR
from src.utils.visualization import (
    load_quantization_reports,
    plot_quantization_ablation_dashboard,
    plot_quantization_curves,
)

EVAL_SPLIT = "val"
REPORTS_DIR = QUANTIZATION_DIR / "reports" / EVAL_SPLIT

results_df, metadata = load_quantization_reports(QUANTIZATION_DIR, eval_split=EVAL_SPLIT)
print(f"Loaded {len(results_df)} ablation rows for split='{EVAL_SPLIT}' from: {REPORTS_DIR}")

### **Environment, Calibration & Head Protection Provenance**

In [ ]:
env = metadata.get("environment", {})
cal = metadata.get("calibration", {})
onnx_cfg = metadata.get("onnx_quantization", {})
excluded_map = onnx_cfg.get("excluded_nodes_by_resolution", {})

provenance_df = pd.DataFrame([
    {"Item": "Source Weights", "Value": metadata.get("source_weights")},
    {"Item": "Weights SHA256", "Value": metadata.get("source_weights_sha256")},
    {"Item": "Eval Split", "Value": metadata.get("eval_split")},
    {"Item": "Train-Only Calibration Images", "Value": f"{cal.get('actual_images')} (max={cal.get('max_images')}, seed={cal.get('seed')})"},
    {"Item": "ONNX PTQ Config", "Value": f"{onnx_cfg.get('format')} {onnx_cfg.get('activation_type')}/{onnx_cfg.get('weight_type')} per_channel={onnx_cfg.get('per_channel')} reduce_range={onnx_cfg.get('reduce_range')} ({onnx_cfg.get('calibrate_method')})"},
    {"Item": "Excluded ONNX Head Nodes", "Value": ", ".join(f"{k}: {len(v)}" for k, v in excluded_map.items())},
    {"Item": "CPU / Platform", "Value": f"{env.get('platform')} | {env.get('machine')} | cores={env.get('cpu_count')}"},
])
display(provenance_df.style.hide(axis="index"))

### **Stage-by-Stage Size, CPU Speed & Threshold-Free Quality**

In [ ]:
main_cols = [
    "imgsz",
    "stage",
    "size_mb",
    "compute_mean_ms",
    "compute_fps",
    "e2e_mean_ms",
    "e2e_p95_ms",
    "e2e_fps",
    "rss_mb",
    "mAP50",
    "mAP50-95",
    "AP50",
    "AP50_small",
    "Recall_small (raw)",
    "Neg FPR @ R>=0.85 (%)",
    "Mean Neg FA Rate (%)",
    "tp_conf_median",
    "tp_recall_ge_0_30",
]
present_main = [col for col in main_cols if col in results_df.columns]
display(
    results_df[present_main]
    .style
    .format(precision=4, na_rep="-")
    .hide(axis="index")
)

### **Quantization Deltas (FP32 -> INT8 Within Resolution) & Speedups**

In [ ]:
delta_cols = [
    "imgsz",
    "stage",
    "size_ratio_vs_pt",
    "size_reduction_pct_vs_fp32",
    "speedup_compute_vs_fp32",
    "speedup_e2e_vs_pt",
    "ap50_delta_vs_fp32",
    "ap50_small_delta_vs_fp32",
    "map50_delta_vs_fp32",
    "map50_95_delta_vs_fp32",
    "neg_fpr_at_r85_delta_vs_fp32",
]
present_deltas = [col for col in delta_cols if col in results_df.columns]
display(
    results_df[present_deltas]
    .style
    .format(precision=4, na_rep="-")
    .hide(axis="index")
)

### **Scale-Stratified Breakdown (Small / Medium / Large)**

In [ ]:
scale_rows = []
for key, detail in (metadata.get("stage_details") or {}).items():
    stage_name, _, res_str = key.rpartition("_")
    for entry in detail.get("by_scale", []):
        scale_rows.append({"imgsz": int(res_str), "stage": stage_name, **entry})

if scale_rows:
    scale_df = pd.DataFrame(scale_rows).sort_values(["imgsz", "Scale", "stage"], ascending=[False, True, True])
    display(scale_df.style.format(precision=4, na_rep="-").hide(axis="index"))

### **Ablation Dashboard & Threshold-Free Trade-Off Curves**

In [ ]:
plot_quantization_ablation_dashboard(
    results_df,
    metadata,
    save_path=REPORTS_DIR / "quantization_ablation_dashboard.png",
)

In [ ]:
plot_quantization_curves(
    metadata,
    save_path=REPORTS_DIR / "quantization_curves.png",
)